# OCR Quality Inspection Notebook

This notebook is a manual inspection workflow for PolyDocBench OCR experiments.

Use it to:

1. Select an experiment artifact by language, article, template, page, and noise profile.
2. Load one noisy scan and its transformed GT.
3. Run Tesseract on the image.
4. Compute OCR quality metrics against the transformed GT.
5. Visualize transformed GT polygons and Tesseract predictions on the scan.

The notebook is especially useful for checking affine scan noise such as `medium_scan_0.jpg` or `heavy_scan_0.jpg`, where GT coordinates must be compared in image pixel coordinates.


## 1. Environment Setup

Run this notebook from the repository root or from the `notebooks/` directory. The cell below detects the project root and imports PolyDocBench modules.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

from PIL import Image, ImageDraw

try:
    from IPython.display import Image as IPyImage
    from IPython.display import Markdown, display
except ImportError:
    def display(value):
        print(value)

    def Markdown(value):
        return value

    class IPyImage:
        def __init__(self, filename: str):
            self.filename = filename

        def __repr__(self) -> str:
            return self.filename

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 2. Experiment Configuration

The experiment pipeline stores artifacts using this hierarchy:

```text
outputs/experiments/tesseract_quality/<language>/<article>/<template>/noisy/page_<NNN>/<noise_profile>_<variant>.jpg
```

Change the selection below to inspect another result. `NOISE_PROFILE` selects the level of scan noise:

- `light_scan`: resolution, illumination, and compression noise.
- `medium_scan`: includes affine/rotation noise and is recommended for transformed-GT checks.
- `heavy_scan`: stronger scan noise including affine/rotation effects.


In [ ]:
LANGUAGE_CODE = "en"
ARTICLE_ID = "quantum_mechanics"
TEMPLATE = "simple_article"
PAGE_NUMBER = 1
NOISE_PROFILE = "medium_scan"
VARIANT = 0

AVAILABLE_NOISE_PROFILES = ("light_scan", "medium_scan", "heavy_scan")
assert NOISE_PROFILE in AVAILABLE_NOISE_PROFILES, f"Unknown noise profile: {NOISE_PROFILE}"
assert PAGE_NUMBER >= 1, "PAGE_NUMBER is one-based and must be positive"

EXPERIMENT_ROOT = PROJECT_ROOT / "outputs" / "experiments" / "tesseract_quality"
CASE_ROOT = EXPERIMENT_ROOT / LANGUAGE_CODE / ARTICLE_ID / TEMPLATE
PAGE_DIR = f"page_{PAGE_NUMBER:03d}"
ARTIFACT_STEM = f"{NOISE_PROFILE}_{VARIANT}"
SCAN_PATH = CASE_ROOT / "noisy" / PAGE_DIR / f"{ARTIFACT_STEM}.jpg"
GT_PATH = CASE_ROOT / "noisy" / PAGE_DIR / f"{ARTIFACT_STEM}_gt.json"
PREDICTION_PATH = CASE_ROOT / "predictions" / PAGE_DIR / f"{ARTIFACT_STEM}_tesseract.json"
OVERLAY_PATH = CASE_ROOT / "debug" / PAGE_DIR / f"{ARTIFACT_STEM}_quality_overlay.jpg"

# Tesseract language code. Examples: eng, rus, fra, deu, spa, ita.
TESSERACT_LANG_BY_CASE = {
    "en": "eng",
    "ru": "rus",
    "fr": "fra",
    "de": "deu",
    "es": "spa",
    "it": "ita",
}
TESSERACT_LANG = TESSERACT_LANG_BY_CASE.get(LANGUAGE_CODE, "eng")

# If Python cannot find tesseract through PATH, set the executable explicitly.
TESSERACT_CMD = shutil.which("tesseract") or r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print("Article:", ARTICLE_ID)
print("Template:", TEMPLATE)
print("Page:", PAGE_NUMBER)
print("Noise profile:", NOISE_PROFILE)
print("Scan:", SCAN_PATH)
print("GT:", GT_PATH)
print("Predictions:", PREDICTION_PATH)
print("Overlay:", OVERLAY_PATH)
print("Tesseract:", TESSERACT_CMD)
print("Tesseract language:", TESSERACT_LANG)


## 3. Load Scan And Transformed GT

For noisy scans, GT must use image coordinates:

- unit: `pixels`
- origin: `top-left`

The `transform.matrix` describes how source PDF bboxes were mapped into the noisy image. `polygon` stores the transformed four-corner geometry; `bbox` stores the horizontal box around that polygon.

In [ ]:
from polydocbench.eval import extract_gt_lines, load_gt

assert SCAN_PATH.exists(), f"Scan image was not found: {SCAN_PATH}"
assert GT_PATH.exists(), f"GT file was not found: {GT_PATH}"

scan = Image.open(SCAN_PATH).convert("RGB")
gt = load_gt(GT_PATH)
metadata = gt.get("metadata", {})
coordinate_system = metadata.get("coordinate_system", {})
transform = metadata.get("transform", {})

display(Markdown(f"**Image size:** {scan.width} x {scan.height}"))
display(Markdown(f"**GT coordinate system:** `{coordinate_system}`"))
display(Markdown(f"**Transform:** `{transform}`"))

assert coordinate_system.get("unit") == "pixels", "Expected noisy GT in pixel coordinates"
assert coordinate_system.get("origin") == "top-left", "Expected noisy GT with top-left origin"

gt_lines = extract_gt_lines(gt, page_number=1)
print(f"GT lines: {len(gt_lines)}")
print("First GT line:")
print(json.dumps(gt_lines[0], ensure_ascii=False, indent=2) if gt_lines else "No GT lines")

## 4. Run Tesseract

The important detail is `coordinate_system="image"`: Tesseract bboxes are kept in top-left pixel coordinates, so they can be compared directly with noisy GT.

In [ ]:
import pytesseract

from polydocbench.eval.ocr import extract_tesseract_lines

if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = str(TESSERACT_CMD)

ocr_lines = extract_tesseract_lines(
    SCAN_PATH,
    lang=TESSERACT_LANG,
    page_number=1,
    coordinate_system="image",
)

PREDICTION_PATH.parent.mkdir(parents=True, exist_ok=True)
PREDICTION_PATH.write_text(json.dumps(ocr_lines, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"OCR lines: {len(ocr_lines)}")
print(f"Saved predictions: {PREDICTION_PATH}")
print("First OCR line:")
print(json.dumps(ocr_lines[0], ensure_ascii=False, indent=2) if ocr_lines else "No OCR lines")

## 5. Compute OCR Quality Metrics

Metrics are computed with PolyDocBench line matching:

- `CER`: character error rate, lower is better.
- `WER`: word error rate, lower is better.
- `IoU`: average matched bbox overlap, higher is better.
- `matched_ratio`: share of GT lines matched to OCR lines by IoU.

Current quality evaluation is bbox-based. For rotated GT, the evaluator uses the axis-aligned `bbox` that encloses the transformed `polygon`.

In [ ]:
from polydocbench.eval import evaluate_ocr_quality

IOU_THRESHOLD = 0.3

metrics = evaluate_ocr_quality(gt_lines, ocr_lines, iou_threshold=IOU_THRESHOLD)
metrics


## 6. Inspect Line Matches

By default this cell prints all matched GT lines. Set `MAX_MATCHES_TO_PRINT` to an integer if you want a shorter preview.

If the overlay shows OCR boxes farther down the image but this cell does not print them, check this limit first.

In [ ]:
from polydocbench.eval.matching import match_lines

# Use None to print all GT lines. Set, for example, 20 for a short preview.
MAX_MATCHES_TO_PRINT = None

matches = match_lines(gt_lines, ocr_lines, iou_threshold=IOU_THRESHOLD)
matches_to_print = matches if MAX_MATCHES_TO_PRINT is None else matches[:MAX_MATCHES_TO_PRINT]

print(f"Total GT lines: {len(gt_lines)}")
print(f"Total OCR lines: {len(ocr_lines)}")
print(f"Total matches to inspect: {len(matches_to_print)}")

for index, match in enumerate(matches_to_print, start=1):
    pred_text = match.prediction["text"] if match.prediction else "<NO MATCH>"
    print(f"--- Match {index} | IoU={match.iou:.3f}")
    print("GT  :", match.gt["text"])
    print("OCR :", pred_text)


## 7. Visualize GT And OCR Predictions

Color convention:

- Red: transformed GT polygon.
- Green: Tesseract prediction bbox.

For rotated scans, red polygons should follow the rotated line geometry. Tesseract predictions remain horizontal because Tesseract returns axis-aligned boxes.

In [ ]:
def iter_gt_elements(gt_json: dict) -> list[dict]:
    elements = []
    for page in gt_json.get("pages", []):
        for container in page.get("containers", []):
            elements.extend(container.get("elements", []))
    return elements or list(gt_json.get("elements", []))


def draw_quality_overlay(
    image_path: Path,
    gt_json: dict,
    predictions: list[dict],
    output_path: Path,
    draw_gt_polygons: bool = True,
    draw_gt_bboxes: bool = False,
    draw_ocr_bboxes: bool = True,
    line_width: int = 2,
) -> Path:
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    for element in iter_gt_elements(gt_json):
        if draw_gt_polygons and element.get("polygon"):
            points = [tuple(point) for point in element["polygon"]]
            draw.line(points + [points[0]], fill="red", width=line_width)

        if draw_gt_bboxes and element.get("bbox"):
            bbox = element["bbox"]
            x = float(bbox["x"])
            y = float(bbox["y"])
            draw.rectangle(
                [x, y, x + float(bbox["width"]), y + float(bbox["height"])],
                outline="blue",
                width=line_width,
            )

    if draw_ocr_bboxes:
        for line in predictions:
            bbox = line.get("bbox", {})
            if not bbox:
                continue
            x = float(bbox["x"])
            y = float(bbox["y"])
            draw.rectangle(
                [x, y, x + float(bbox["width"]), y + float(bbox["height"])],
                outline="green",
                width=line_width,
            )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path)
    return output_path

overlay_path = draw_quality_overlay(
    SCAN_PATH,
    gt,
    ocr_lines,
    OVERLAY_PATH,
    draw_gt_polygons=True,
    draw_gt_bboxes=False,
    draw_ocr_bboxes=True,
)

print(f"Saved overlay: {overlay_path}")
display(IPyImage(filename=str(overlay_path)))

## 8. Optional: Draw Separate Debug Layers

Use these cells when the combined overlay is too visually dense. The separate outputs keep the same convention: red for GT polygons and green for Tesseract predictions.

In [ ]:
gt_polygon_overlay = OVERLAY_PATH.with_name(f"{ARTIFACT_STEM}_gt_polygons.jpg")
ocr_bbox_overlay = OVERLAY_PATH.with_name(f"{ARTIFACT_STEM}_ocr_bboxes.jpg")

draw_quality_overlay(
    SCAN_PATH,
    gt,
    [],
    gt_polygon_overlay,
    draw_gt_polygons=True,
    draw_gt_bboxes=False,
    draw_ocr_bboxes=False,
)
draw_quality_overlay(
    SCAN_PATH,
    gt,
    ocr_lines,
    ocr_bbox_overlay,
    draw_gt_polygons=False,
    draw_gt_bboxes=False,
    draw_ocr_bboxes=True,
)

print("GT polygons:", gt_polygon_overlay)
print("OCR bboxes:", ocr_bbox_overlay)


## 9. Sanity Checks For Rotated Scans

Use this section to confirm that `medium_scan` or `heavy_scan` GT was actually transformed.


In [ ]:
matrix = transform.get("matrix", [])
identity_matrix = [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]
non_identity_transform = matrix != identity_matrix

elements = iter_gt_elements(gt)
polygon_count = sum(1 for element in elements if element.get("polygon"))
source_bbox_count = sum(1 for element in elements if element.get("metadata", {}).get("source_bbox"))

print("Transform matrix:", matrix)
print("Non-identity transform:", non_identity_transform)
print("Elements:", len(elements))
print("Elements with polygon:", polygon_count)
print("Elements with source_bbox:", source_bbox_count)

if NOISE_PROFILE in {"medium_scan", "heavy_scan"}:
    assert non_identity_transform, "Expected a non-identity transform for affine/rotation noise profile"

assert polygon_count == len(elements), "Expected every GT element to have transformed polygon geometry"
assert source_bbox_count == len(elements), "Expected every GT element to preserve source_bbox metadata"
